In [1]:
# 第1步：安装依赖（在 notebook 中运行）
import sys
import subprocess

# ultralytics 的跟踪功能需要 lapx
packages = [
    'ultralytics',
    'opencv-python',
    'fastapi',
    'uvicorn',
    'numpy',
    'lapx' 
]

def pip_install(pkgs):
    for p in pkgs:
        try:
            print(f'Installing {p}...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
            print(f'✓ {p} installed')
        except subprocess.CalledProcessError as e:
            print(f'✗ Failed to install {p}: {e}')

pip_install(packages)
print('\n✅ Dependency installation complete')

Installing ultralytics...
✓ ultralytics installed
Installing opencv-python...
✓ ultralytics installed
Installing opencv-python...
✓ opencv-python installed
Installing fastapi...
✓ opencv-python installed
Installing fastapi...
✓ fastapi installed
Installing uvicorn...
✓ fastapi installed
Installing uvicorn...
✓ uvicorn installed
Installing numpy...
✓ uvicorn installed
Installing numpy...
✓ numpy installed
Installing lapx...
✓ numpy installed
Installing lapx...
✓ lapx installed

✅ Dependency installation complete
✓ lapx installed

✅ Dependency installation complete


In [2]:
# 第2步：导入库
import threading
import time
from typing import List, Dict
import numpy as np
import cv2
from ultralytics import YOLO

# 共享变量：当前跟踪到的行人信息（由视频线程维护）
current_tracks = []  # list of dicts: {id, x1, y1, x2, y2, score}
model = None

print('✅ Libraries imported')

✅ Libraries imported


In [3]:
# 第3步：初始化 YOLO 模型
def init_model(model_name='yolov8n.pt'):
    global model
    print(f'Loading YOLO model: {model_name}...')
    model = YOLO(model_name)
    print('✓ YOLO model loaded')
    print('✓ Will use built-in ByteTrack for tracking')

# 调用初始化
init_model('yolov8n.pt')

Loading YOLO model: yolov8n.pt...
✓ YOLO model loaded
✓ Will use built-in ByteTrack for tracking


In [4]:
# 第4步：视频读取和检测线程
def video_loop(source=0, display=True, conf_thresh=0.3):
    global current_tracks, model
    
    # 尝试打开摄像头
    print(f'Attempting to open camera source {source}...')
    cap = cv2.VideoCapture(source)
    
    # 如果 source 0 失败，尝试 source 1 (有时 macOS 上 1 是前置)
    if not cap.isOpened() and source == 0:
        print('Source 0 failed, trying source 1...')
        cap = cv2.VideoCapture(1)
        source = 1

    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera (source={source}). Please check permissions.')
    
    print(f'✓ Camera opened (source={source})')
    print('Press Q in video window or CTRL-C to stop')
    
    frame_count = 0
    fail_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                fail_count += 1
                print(f'Warning: Failed to read frame (count={fail_count})')
                if fail_count > 10:
                    print('Too many failed reads, exiting loop.')
                    break
                time.sleep(0.1)
                continue
            
            fail_count = 0 # reset on success
            frame_count += 1
            
            # 使用 YOLOv8 内置的 ByteTrack 进行跟踪
            # tracker="bytetrack.yaml" 确保使用 ByteTrack
            results = model.track(
                source=frame, 
                persist=True, 
                tracker="bytetrack.yaml", 
                conf=conf_thresh, 
                classes=[0],
                verbose=False
            )
            
            tracks_out = []
            
            if results and results[0].boxes and results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                track_ids = results[0].boxes.id.int().cpu().numpy()
                scores = results[0].boxes.conf.cpu().numpy()
                
                for box, track_id, score in zip(boxes, track_ids, scores):
                    x1, y1, x2, y2 = box
                    tracks_out.append({
                        'id': int(track_id),
                        'x1': float(x1),
                        'y1': float(y1),
                        'x2': float(x2),
                        'y2': float(y2),
                        'score': float(score)
                    })
            
            current_tracks = tracks_out
            
            if display:
                vis = frame.copy()
                # Draw tracks
                for t in tracks_out:
                    x1, y1, x2, y2 = int(t['x1']), int(t['y1']), int(t['x2']), int(t['y2'])
                    tid = t['id']
                    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(vis, f'ID:{tid}', (x1, max(10, y1 - 5)),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Add status text
                cv2.putText(vis, f'Frame: {frame_count}', (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(vis, 'Tracker: ByteTrack', (10, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                
                cv2.imshow('Webcam - YOLO + ByteTrack', vis)
                
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print('Q pressed, exiting...')
                    break
            
            time.sleep(0.001)
    
    except KeyboardInterrupt:
        print('✓ Stopping video loop (KeyboardInterrupt)')
    except Exception as e:
        print(f'Error in video loop: {e}')
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)

print('✅ Video loop function updated (Displaying "Tracker: ByteTrack")')

✅ Video loop function updated (Displaying "Tracker: ByteTrack")


In [5]:
# 第5步：FastAPI 服务
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get('/current_pedestrians')
def get_current_pedestrians():
    """返回当前视频中跟踪到的唯一行人及其边界框"""
    global current_tracks
    return {
        'pedestrians': current_tracks,
        'count': len(current_tracks)
    }

def start_api(host='0.0.0.0', port=8000):
    """在后台线程中启动 FastAPI 服务"""
    def run():
        uvicorn.run(app, host=host, port=port, log_level='warning')
    
    t = threading.Thread(target=run, daemon=True)
    t.start()
    print(f'✅ API started at http://{host}:{port}/current_pedestrians')

print('✅ FastAPI service defined')

✅ FastAPI service defined


In [ ]:
# 第6步：启动系统
# 注意：在 macOS 上，OpenCV 的 imshow 窗口必须在主线程中运行。

# 1. 启动 API (后台线程)
start_api(host='0.0.0.0', port=8000)

print('\n' + '='*60)
print('🚀 系统启动中...')
print('🌐 API 地址: http://localhost:8000/current_pedestrians')
print('📹 正在尝试打开摄像头...')
print('⚠️  如果看不到窗口，请检查下方输出是否有错误信息。')
print('⚠️  按 Q 键退出视频窗口。')
print('='*60)

# 2. 启动视频循环 (主线程)
try:
    # 尝试 source 0，如果失败代码内部会尝试 source 1
    video_loop(source=0, display=True, conf_thresh=0.3)
except KeyboardInterrupt:
    print("已通过键盘中断停止")
except Exception as e:
    print(f"运行时错误: {e}")
    
print("视频循环已结束")

✅ API started at http://0.0.0.0:8000/current_pedestrians

🚀 系统启动中...
🌐 API 地址: http://localhost:8000/current_pedestrians
📹 正在尝试打开摄像头...
⚠️  如果看不到窗口，请检查下方输出是否有错误信息。
⚠️  按 Q 键退出视频窗口。
Attempting to open camera source 0...
✓ Camera opened (source=0)
Press Q in video window or CTRL-C to stop
✓ Camera opened (source=0)
Press Q in video window or CTRL-C to stop
✓ Stopping video loop (KeyboardInterrupt)
视频循环已结束
✓ Stopping video loop (KeyboardInterrupt)
视频循环已结束


: 

## 使用说明

### 运行步骤
1. 依次执行上面的单元格（第1步 → 第6步）
2. 第1步会安装所需依赖（需要几分钟）
3. 第4步会下载 YOLO 模型权重（首次运行）
4. 第6步会打开摄像头窗口并启动 API 服务

### API 接口
- **端点**: `GET http://localhost:8000/current_pedestrians`
- **返回**: JSON 格式，包含当前跟踪到的唯一行人
```json
{
  "pedestrians": [
    {"id": 1, "x1": 100.5, "y1": 200.3, "x2": 300.2, "y2": 500.8, "score": 0.95},
    {"id": 2, "x1": 400.1, "y1": 150.2, "x2": 550.3, "y2": 480.5, "score": 0.87}
  ],
  "count": 2
}
```

### 测试命令
```bash
# 查看当前检测到的行人
curl http://localhost:8000/current_pedestrians

# 格式化输出
curl http://localhost:8000/current_pedestrians | python -m json.tool
```

### 注意事项
- macOS 首次使用可能需要授权摄像头权限
- 如果 ByteTrack 安装失败，会自动使用 IoU 回退追踪器
- 视频窗口中按 **Q** 键可关闭窗口
- 每个行人都有唯一的 `id`，在跟踪期间保持不变

In [ ]:
# 测试：检查 current_tracks 和 API 路由
import requests
import time

print(f'当前 current_tracks: {current_tracks}')
print(f'当前 current_tracks 数量: {len(current_tracks)}')

# 测试 API
time.sleep(2)
try:
    response = requests.get('http://localhost:8000/current_pedestrians')
    print(f'\nAPI 响应状态码: {response.status_code}')
    print(f'API 响应内容: {response.json()}')
except Exception as e:
    print(f'API 请求失败: {e}')
    
# 检查 FastAPI 的所有路由
print(f'\n注册的路由:')
for route in app.routes:
    print(f'  {route.path} - {route.methods if hasattr(route, "methods") else "N/A"}')

当前 current_tracks: []
当前 current_tracks 数量: 0


/Users/chenshi/Desktop/ipynotebook/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



API 响应状态码: 404
API 响应内容: {'detail': 'Not Found'}

注册的路由:
  /openapi.json - {'GET', 'HEAD'}
  /docs - {'GET', 'HEAD'}
  /docs/oauth2-redirect - {'GET', 'HEAD'}
  /redoc - {'GET', 'HEAD'}
  /current_pedestrians - {'GET'}
